# No-progress sensitivity analysis
## EV-battery next-operation ranking without sequence-progress input

This notebook is a sensitivity analysis of the final topology-mechanism experiment. It removes the scalar sequence-progress feature $p_t=t/(T-1)$ from the pairwise scoring head while keeping the remaining experimental design fixed.

### Purpose

The main manuscript uses normalized sequence progress as one input to the pairwise scorer. Because this quantity depends on the length of the held-out reference sequence, this sensitivity analysis asks whether the central topology-mechanism pattern remains when that feature is unavailable.

### Conditions

The notebook evaluates the same strict 10-fold leave-one-product-out protocol with neural-training seeds `[17, 29, 43]`:

1. **RealGNN** — recorded physical fixation topology.
2. **NoEdgesGNN** — the graph branch is retained but all cross-node message-passing edges are removed.
3. **ShuffledGNN × 5** — five semantic–topology permutations using the same fixed permutation construction as the main experiment.
4. **MatchedMLP** — a non-graph MLP whose hidden width is selected separately within each LOPO fold to match the no-progress GNN parameter count as closely as possible.

All other training settings are retained: 30 epochs, AdamW, eight sampled negatives per training state, the same learning rate, weight decay, dropout, categorical embeddings, graph dimensions, permutation seeds, and tie-aware evaluation.

### Interpretation

This is a **sensitivity analysis**, not a replacement for the canonical experiment. The main question is whether the qualitative mechanism result survives without sequence progress:

- RealGNN > NoEdgesGNN,
- permutation mean > NoEdgesGNN,
- and no demonstrated RealGNN advantage over the permutation mean.

The notebook writes all outputs to a new directory and does not overwrite the canonical `final_topology_mechanism_*` files.


In [ ]:
# 1. Imports and frozen configuration

from pathlib import Path
from collections import Counter, defaultdict
from itertools import product
import tarfile, json, math, random, time

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

SEEDS = [17, 29, 43]
SHUFFLE_IDS = [1, 2, 3, 4, 5]

EPOCHS = 30
NEGATIVES_PER_STATE = 8
LR = 2e-3
WEIGHT_DECAY = 1e-4
DROPOUT = 0.10

EMBED_DIM = 12
HIDDEN_DIM = 64
GNN_DIM = 32

MATCHED_WIDTH_CANDIDATES = list(range(64, 257, 2))

BASE_SEED = 20261003
GRAPH_SHUFFLE_BASE = 20261119

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Per fold/seed: Real + No-edge + five permutations + one matched MLP.
N_FITS = 10 * len(SEEDS) * (1 + 1 + len(SHUFFLE_IDS) + 1)

print("Device:", DEVICE)
print("Torch:", torch.__version__)
print("Planned neural fits:", N_FITS)

if DEVICE.type != "cuda":
    print("WARNING: GPU is strongly recommended for this notebook.")


In [ ]:
# 2. Find the KIT dataset automatically.
#
# The cell first searches for already-extracted CSVs anywhere under /content.
# If not found, it searches for the uploaded KIT .tar archive and extracts it.

root = Path("/content")

catalogues = list(root.rglob("battery_catalogue.csv"))

if catalogues:
    DATA = catalogues[0].parent
else:
    archives = list(root.rglob("10.35097-emz24pksshndq468.tar"))
    if not archives:
        archives = list(root.rglob("10.35097-*.tar"))

    assert archives, (
        "KIT dataset not found. Upload 10.35097-emz24pksshndq468.tar "
        "to the Colab Files panel, then rerun this cell."
    )

    TAR_PATH = archives[0]
    print("Using archive:", TAR_PATH)

    work = root / "battery_disassembly_dataset_extracted"
    work.mkdir(parents=True, exist_ok=True)

    with tarfile.open(TAR_PATH, "r") as tf:
        tf.extractall(work)

    catalogues = list(work.rglob("battery_catalogue.csv"))
    assert catalogues, "battery_catalogue.csv not found after extraction."
    DATA = catalogues[0].parent

print("Dataset directory:", DATA)

In [ ]:
# 3. Load and validate raw data.

bat = pd.read_csv(DATA / "battery_catalogue.csv")
parts = pd.read_csv(DATA / "parts.csv")
fix = pd.read_csv(DATA / "fixations.csv")
ops = pd.read_csv(DATA / "operations.csv")

print("Batteries:", len(bat))
print("Parts:", len(parts))
print("Fixations:", len(fix))
print("Operations:", len(ops))

display(bat[["id", "name"]])

In [ ]:
# 4. Attach each operation to a battery and reconstruct the 10 reference chains.

battery_names = bat.set_index("id")["name"].to_dict()

fix_battery = fix.set_index("id")["battery_id"].to_dict()
part_battery = parts.set_index("id")["battery_id"].to_dict()
part_class_raw = parts.set_index("id")["class"].to_dict()

fix_info_raw = (
    fix.set_index("id")[["class", "fromPartID", "toPartID"]]
    .to_dict("index")
)

def operation_battery(row):
    if pd.notna(row.fixation_id):
        return int(fix_battery[int(row.fixation_id)])
    return int(part_battery[int(row.part_id)])

ops["battery_id"] = ops.apply(operation_battery, axis=1)
ops_by_id = ops.set_index("id")

chains = {}

for b in sorted(ops["battery_id"].unique()):
    sub = ops[ops["battery_id"] == b]
    ids = set(sub["id"].astype(int))
    next_ids = set(sub["next_op_id"].dropna().astype(int))

    starts = list(ids - next_ids)
    terminals = sub[sub["next_op_id"].isna()]["id"].astype(int).tolist()

    assert len(starts) == 1
    assert len(terminals) == 1

    seq = []
    seen = set()
    cur = int(starts[0])

    while cur in ids and cur not in seen:
        seen.add(cur)
        seq.append(cur)

        nxt = ops_by_id.loc[cur, "next_op_id"]
        if pd.isna(nxt):
            break
        cur = int(nxt)

    assert len(seq) == len(ids), f"Broken chain for battery {b}"
    chains[int(b)] = seq

print({
    battery_names[b]: len(seq)
    for b, seq in chains.items()
})

In [ ]:
# 5. Raw full relational signatures.
#
# These are used only for diagnostics (seen vs unseen motif/signature).
# They are NOT output classes for the neural ranking models.

def raw_full_signature(op_id):
    r = ops_by_id.loc[int(op_id)]
    skill = str(r.skill_id)

    if pd.notna(r.fixation_id):
        fi = fix_info_raw[int(r.fixation_id)]
        src_cls = str(part_class_raw[int(fi["fromPartID"])])
        dst_cls = str(part_class_raw[int(fi["toPartID"])])
        return (
            f"{skill}|fix:{fi['class']}|"
            f"{src_cls}->{dst_cls}"
        )

    return (
        f"{skill}|part:"
        f"{part_class_raw[int(r.part_id)]}"
    )

raw_signatures = {
    b: [raw_full_signature(op_id) for op_id in seq]
    for b, seq in chains.items()
}

In [ ]:
# 6. Fold-specific vocabularies.
#
# Vocabularies are learned ONLY from the 9 training batteries.
# Category 0 is <UNK>, so truly unseen test categories do not leak into training.

def make_vocab(values):
    vals = sorted({
        str(x)
        for x in values
        if pd.notna(x)
    })
    return {
        "<UNK>": 0,
        **{v: i + 1 for i, v in enumerate(vals)}
    }

def lookup(vocab, value):
    if pd.isna(value):
        return 0
    return vocab.get(str(value), 0)

def build_fold_vocab(train_batteries):
    return {
        "part": make_vocab(
            parts[
                parts["battery_id"].isin(train_batteries)
            ]["class"]
        ),
        "fix": make_vocab(
            fix[
                fix["battery_id"].isin(train_batteries)
            ]["class"]
        ),
        "skill": make_vocab(
            ops[
                ops["battery_id"].isin(train_batteries)
            ]["skill_id"]
        ),
    }

In [ ]:
# 7. Convert one battery into product-invariant graph + operation features.
#
# graph_mode:
#   "real"    -> true physical fixation topology
#   "shuffle" -> fixed permutation of topology relative to semantic/operation nodes
#   "noedge"  -> same GNN architecture, but no message-passing edges
#
# Operation-local attributes are identical across all graph modes.
# Therefore only the graph branch is manipulated.

def build_battery_pack(
    battery_id,
    vocab,
    graph_mode="real",
    shuffle_id=0,
):
    assert graph_mode in {"real", "shuffle", "noedge"}

    psub = parts[
        parts["battery_id"] == battery_id
    ].copy()

    fsub = fix[
        fix["battery_id"] == battery_id
    ].copy()

    part_id_to_local = {
        int(pid): i
        for i, pid in enumerate(psub["id"].astype(int))
    }

    n_nodes = len(psub)

    true_degree = np.zeros(n_nodes, dtype=np.float32)

    edge_src = []
    edge_dst = []
    edge_type = []

    for _, r in fsub.iterrows():
        a = part_id_to_local[int(r.fromPartID)]
        b = part_id_to_local[int(r.toPartID)]

        true_degree[a] += 1
        true_degree[b] += 1

        # Undirected message-passing representation.
        edge_src += [a, b]
        edge_dst += [b, a]

        et = lookup(vocab["fix"], r["class"])
        edge_type += [et, et]

    true_degree_norm = (
        true_degree / max(1.0, float(true_degree.max()))
    )

    node_part_class = np.array(
        [
            lookup(vocab["part"], cls)
            for cls in psub["class"]
        ],
        dtype=np.int64,
    )

    true_edge_index = np.array(
        [edge_src, edge_dst],
        dtype=np.int64,
    )

    true_edge_type = np.array(
        edge_type,
        dtype=np.int64,
    )

    if graph_mode == "real":
        graph_edge_index = true_edge_index.copy()
        graph_edge_type = true_edge_type.copy()
        graph_degree_norm = true_degree_norm.copy()

    elif graph_mode == "shuffle":
        # A deterministic, independent permutation for each
        # (shuffle_id, battery). This preserves the unlabeled graph,
        # but reassigns its physical locations to semantic nodes.
        rng_graph = np.random.default_rng(
            GRAPH_SHUFFLE_BASE
            + 10007 * int(shuffle_id)
            + int(battery_id)
        )

        permutation = rng_graph.permutation(n_nodes)

        graph_edge_index = permutation[true_edge_index]
        graph_edge_type = true_edge_type.copy()

        shuffled_degree = np.bincount(
            graph_edge_index.reshape(-1),
            minlength=n_nodes,
        ).astype(np.float32)

        graph_degree_norm = (
            shuffled_degree
            / max(1.0, float(shuffled_degree.max()))
        )

    else:  # noedge
        graph_edge_index = np.empty((2, 0), dtype=np.int64)
        graph_edge_type = np.empty((0,), dtype=np.int64)

        # Do not leak adjacency through the graph branch.
        graph_degree_norm = np.zeros(
            n_nodes,
            dtype=np.float32,
        )

    seq = [int(x) for x in chains[battery_id]]

    categorical = []
    graph_nodes = []
    numeric = []
    signatures = []

    for op_id in seq:
        r = ops_by_id.loc[op_id]

        skill_idx = lookup(
            vocab["skill"],
            r.skill_id,
        )

        if pd.notna(r.fixation_id):
            fi = fix_info_raw[int(r.fixation_id)]

            src_pid = int(fi["fromPartID"])
            dst_pid = int(fi["toPartID"])

            src_node = part_id_to_local[src_pid]
            dst_node = part_id_to_local[dst_pid]

            categorical.append([
                skill_idx,
                0,
                lookup(vocab["fix"], fi["class"]),
                lookup(vocab["part"], part_class_raw[src_pid]),
                lookup(vocab["part"], part_class_raw[dst_pid]),
                0,
            ])

            graph_nodes.append([
                src_node,
                dst_node,
                -1,
            ])

            # IMPORTANT:
            # identical operation-local features in every graph condition.
            numeric.append([
                true_degree_norm[src_node],
                true_degree_norm[dst_node],
                0.0,
            ])

        else:
            pid = int(r.part_id)
            node = part_id_to_local[pid]

            categorical.append([
                skill_idx,
                1,
                0,
                0,
                0,
                lookup(vocab["part"], part_class_raw[pid]),
            ])

            graph_nodes.append([
                -1,
                -1,
                node,
            ])

            numeric.append([
                0.0,
                0.0,
                true_degree_norm[node],
            ])

        signatures.append(raw_full_signature(op_id))

    return {
        "battery_id": int(battery_id),
        "battery_name": battery_names[int(battery_id)],
        "node_part_class": node_part_class,
        "node_degree": graph_degree_norm.astype(np.float32),
        "edge_index": graph_edge_index,
        "edge_type": graph_edge_type,
        "op_categorical": np.asarray(categorical, dtype=np.int64),
        "op_graph_nodes": np.asarray(graph_nodes, dtype=np.int64),
        "op_numeric": np.asarray(numeric, dtype=np.float32),
        "full_signatures": signatures,
        "n_ops": len(seq),
        "graph_mode": graph_mode,
        "shuffle_id": int(shuffle_id),
    }

In [ ]:
# 8. Neural models.

class BaseOperationRanker(nn.Module):
    """
    Product-invariant MLP operation ranker.

    It never receives battery/product IDs or raw entity IDs.
    """

    def __init__(
        self,
        n_skill,
        n_fix,
        n_part,
        emb_dim=EMBED_DIM,
        hidden_dim=HIDDEN_DIM,
        dropout=DROPOUT,
    ):
        super().__init__()

        self.skill_emb = nn.Embedding(n_skill, emb_dim)
        self.kind_emb = nn.Embedding(2, 4)
        self.fix_emb = nn.Embedding(n_fix, emb_dim)
        self.part_emb = nn.Embedding(n_part, emb_dim)

        op_input_dim = (
            emb_dim      # skill
            + 4          # operation kind
            + emb_dim    # fixation class
            + emb_dim    # source part class
            + emb_dim    # target part class
            + emb_dim    # removed part class
            + 3          # normalized local degrees
        )

        self.op_mlp = nn.Sequential(
            nn.Linear(op_input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
        )

        # current, candidate, abs difference, product
        # Sequence progress is intentionally omitted in this sensitivity run.
        self.score_mlp = nn.Sequential(
            nn.Linear(hidden_dim * 4, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    def base_operation_encoding(self, pack):
        cats = torch.as_tensor(
            pack["op_categorical"],
            dtype=torch.long,
            device=DEVICE,
        )

        nums = torch.as_tensor(
            pack["op_numeric"],
            dtype=torch.float32,
            device=DEVICE,
        )

        skill = cats[:, 0]
        kind = cats[:, 1]
        fix_class = cats[:, 2]
        src_class = cats[:, 3]
        dst_class = cats[:, 4]
        part_class = cats[:, 5]

        x = torch.cat([
            self.skill_emb(skill),
            self.kind_emb(kind),
            self.fix_emb(fix_class),
            self.part_emb(src_class),
            self.part_emb(dst_class),
            self.part_emb(part_class),
            nums,
        ], dim=-1)

        return self.op_mlp(x)

    def encode_operations(self, pack):
        return self.base_operation_encoding(pack)

    def score_pairs(self, current_repr, candidate_repr):
        x = torch.cat([
            current_repr,
            candidate_repr,
            torch.abs(current_repr - candidate_repr),
            current_repr * candidate_repr,
        ], dim=-1)

        return self.score_mlp(x).squeeze(-1)


class GraphEncoder(nn.Module):
    """
    Two-layer product-invariant message-passing network.
    """

    def __init__(
        self,
        n_part,
        n_fix,
        part_emb_dim=EMBED_DIM,
        edge_emb_dim=8,
        graph_dim=GNN_DIM,
        dropout=DROPOUT,
    ):
        super().__init__()

        self.part_emb = nn.Embedding(n_part, part_emb_dim)
        self.edge_emb = nn.Embedding(n_fix, edge_emb_dim)

        self.input_proj = nn.Linear(
            part_emb_dim + 1,
            graph_dim,
        )

        self.message_layers = nn.ModuleList([
            nn.Linear(
                graph_dim + edge_emb_dim,
                graph_dim,
            )
            for _ in range(2)
        ])

        self.self_layers = nn.ModuleList([
            nn.Linear(graph_dim, graph_dim)
            for _ in range(2)
        ])

        self.dropout = dropout

    def forward(self, pack):
        node_class = torch.as_tensor(
            pack["node_part_class"],
            dtype=torch.long,
            device=DEVICE,
        )

        degree = torch.as_tensor(
            pack["node_degree"],
            dtype=torch.float32,
            device=DEVICE,
        ).unsqueeze(-1)

        h = F.relu(
            self.input_proj(
                torch.cat([
                    self.part_emb(node_class),
                    degree,
                ], dim=-1)
            )
        )

        edge_index = torch.as_tensor(
            pack["edge_index"],
            dtype=torch.long,
            device=DEVICE,
        )

        edge_type = torch.as_tensor(
            pack["edge_type"],
            dtype=torch.long,
            device=DEVICE,
        )

        src = edge_index[0]
        dst = edge_index[1]

        edge_embedding = self.edge_emb(edge_type)

        for message_layer, self_layer in zip(
            self.message_layers,
            self.self_layers,
        ):
            msg = message_layer(
                torch.cat([
                    h[src],
                    edge_embedding,
                ], dim=-1)
            )

            agg = torch.zeros(
                (h.shape[0], h.shape[1]),
                dtype=h.dtype,
                device=DEVICE,
            )

            agg.index_add_(0, dst, msg)

            counts = torch.zeros(
                (h.shape[0], 1),
                dtype=h.dtype,
                device=DEVICE,
            )

            counts.index_add_(
                0,
                dst,
                torch.ones(
                    (dst.numel(), 1),
                    dtype=h.dtype,
                    device=DEVICE,
                ),
            )

            agg = agg / counts.clamp_min(1.0)

            h = F.relu(
                self_layer(h) + agg
            )

            h = F.dropout(
                h,
                p=self.dropout,
                training=self.training,
            )

        return h


class GNNOperationRanker(BaseOperationRanker):

    def __init__(
        self,
        n_skill,
        n_fix,
        n_part,
        emb_dim=EMBED_DIM,
        hidden_dim=HIDDEN_DIM,
        graph_dim=GNN_DIM,
        dropout=DROPOUT,
    ):
        super().__init__(
            n_skill=n_skill,
            n_fix=n_fix,
            n_part=n_part,
            emb_dim=emb_dim,
            hidden_dim=hidden_dim,
            dropout=dropout,
        )

        self.graph_encoder = GraphEncoder(
            n_part=n_part,
            n_fix=n_fix,
            part_emb_dim=emb_dim,
            graph_dim=graph_dim,
            dropout=dropout,
        )

        # src embedding, dst embedding, absolute difference
        self.structural_projection = nn.Sequential(
            nn.Linear(graph_dim * 3, hidden_dim),
            nn.ReLU(),
        )

        self.combine = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
        )

    def encode_operations(self, pack):
        base = self.base_operation_encoding(pack)
        node_h = self.graph_encoder(pack)

        op_nodes = torch.as_tensor(
            pack["op_graph_nodes"],
            dtype=torch.long,
            device=DEVICE,
        )

        n_ops = op_nodes.shape[0]

        a = torch.zeros(
            (n_ops, node_h.shape[1]),
            device=DEVICE,
        )

        b = torch.zeros_like(a)

        fixation_mask = op_nodes[:, 0] >= 0

        if fixation_mask.any():
            a[fixation_mask] = node_h[
                op_nodes[fixation_mask, 0]
            ]

            b[fixation_mask] = node_h[
                op_nodes[fixation_mask, 1]
            ]

        part_mask = op_nodes[:, 2] >= 0

        if part_mask.any():
            a[part_mask] = node_h[
                op_nodes[part_mask, 2]
            ]

        structural = self.structural_projection(
            torch.cat([
                a,
                b,
                torch.abs(a - b),
            ], dim=-1)
        )

        return self.combine(
            torch.cat([
                base,
                structural,
            ], dim=-1)
        )

In [ ]:
# 9. Reproducibility + pairwise-ranking training without sequence progress

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def make_training_pairs(n_ops, k, rng):
    current, positive, negative = [], [], []

    for t in range(n_ops - 2):
        pool = np.arange(
            t + 2,
            n_ops,
            dtype=np.int64,
        )

        neg = rng.choice(
            pool,
            size=k,
            replace=(len(pool) < k),
        )

        current.append(t)
        positive.append(t + 1)
        negative.append(neg)

    return (
        np.asarray(current, dtype=np.int64),
        np.asarray(positive, dtype=np.int64),
        np.stack(negative),
    )


def build_gnn(vocab):
    model = GNNOperationRanker(
        n_skill=len(vocab["skill"]),
        n_fix=len(vocab["fix"]),
        n_part=len(vocab["part"]),
    )
    return model.to(DEVICE)


def build_mlp(vocab, hidden_dim):
    model = BaseOperationRanker(
        n_skill=len(vocab["skill"]),
        n_fix=len(vocab["fix"]),
        n_part=len(vocab["part"]),
        hidden_dim=hidden_dim,
    )
    return model.to(DEVICE)


def fit_ranker(
    model,
    packs,
    train_batteries,
    seed,
):
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY,
    )

    # The same seed keeps initialization, battery order, and negative
    # sampling paired as closely as possible across conditions.
    rng = np.random.default_rng(seed)

    for epoch in range(EPOCHS):
        order = list(train_batteries)
        rng.shuffle(order)

        model.train()

        for b in order:
            pack = packs[b]

            cur, pos, neg = make_training_pairs(
                pack["n_ops"],
                NEGATIVES_PER_STATE,
                rng,
            )

            z = model.encode_operations(pack)

            cur_t = torch.as_tensor(
                cur,
                dtype=torch.long,
                device=DEVICE,
            )

            pos_t = torch.as_tensor(
                pos,
                dtype=torch.long,
                device=DEVICE,
            )

            neg_t = torch.as_tensor(
                neg,
                dtype=torch.long,
                device=DEVICE,
            )

            cur_z = z[cur_t]
            pos_z = z[pos_t]

            pos_score = model.score_pairs(
                cur_z,
                pos_z,
            )

            k = neg_t.shape[1]

            cur_rep = (
                cur_z[:, None, :]
                .expand(-1, k, -1)
                .reshape(-1, cur_z.shape[1])
            )

            neg_z = z[neg_t.reshape(-1)]

            neg_score = model.score_pairs(
                cur_rep,
                neg_z,
            ).view(len(cur), k)

            loss = F.softplus(
                -(pos_score[:, None] - neg_score)
            ).mean()

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    return model


def train_one_gnn_condition(
    train_batteries,
    test_battery,
    seed,
    graph_mode,
    shuffle_id=0,
):
    set_seed(seed)

    vocab = build_fold_vocab(train_batteries)

    packs = {
        b: build_battery_pack(
            b,
            vocab,
            graph_mode=graph_mode,
            shuffle_id=shuffle_id,
        )
        for b in train_batteries + [test_battery]
    }

    model = build_gnn(vocab)

    model = fit_ranker(
        model,
        packs,
        train_batteries,
        seed,
    )

    return model, packs[test_battery], vocab


def train_one_mlp_condition(
    train_batteries,
    test_battery,
    seed,
    hidden_dim,
):
    set_seed(seed)

    vocab = build_fold_vocab(train_batteries)

    # The MLP ignores the graph branch. graph_mode="real" is used only
    # to construct the same operation-local attributes, including the
    # fixed normalized degree features used in every condition.
    packs = {
        b: build_battery_pack(
            b,
            vocab,
            graph_mode="real",
            shuffle_id=0,
        )
        for b in train_batteries + [test_battery]
    }

    model = build_mlp(
        vocab,
        hidden_dim=hidden_dim,
    )

    model = fit_ranker(
        model,
        packs,
        train_batteries,
        seed,
    )

    return model, packs[test_battery], vocab


In [ ]:
# 10. Tie-aware ranking metrics.

def expected_tie_metrics(scores, target_position=0, tol=1e-8):
    """
    Scores correspond to candidate operations.

    Returns the expected Hits@1, Hits@3, and reciprocal rank
    under random ordering within exact/near score ties.
    """

    scores = np.asarray(scores)
    target_score = scores[target_position]

    greater = int(
        np.sum(scores > target_score + tol)
    )

    equal = int(
        np.sum(
            np.abs(scores - target_score) <= tol
        )
    )

    equal = max(equal, 1)

    hits1 = (
        max(
            0,
            min(equal, 1 - greater),
        )
        / equal
        if greater < 1
        else 0.0
    )

    hits3 = (
        max(
            0,
            min(equal, 3 - greater),
        )
        / equal
        if greater < 3
        else 0.0
    )

    ranks = np.arange(
        greater + 1,
        greater + equal + 1,
        dtype=float,
    )

    mrr = float(
        np.mean(1.0 / ranks)
    )

    return hits1, hits3, mrr

In [ ]:
# 11. Evaluate one neural model on an unseen battery.

def training_signature_set(train_batteries):
    s = set()

    for b in train_batteries:
        s.update(raw_signatures[b])

    return s


def evaluate_neural_model(
    model,
    test_pack,
    train_batteries,
):
    model.eval()

    seen_signatures = training_signature_set(
        train_batteries
    )

    rows = []

    with torch.no_grad():
        operation_repr = model.encode_operations(
            test_pack
        )

        n_ops = test_pack["n_ops"]

        for t in range(n_ops - 1):
            candidate_indices = np.arange(
                t + 1,
                n_ops,
                dtype=np.int64,
            )

            candidate_t = torch.as_tensor(
                candidate_indices,
                dtype=torch.long,
                device=DEVICE,
            )

            current_repr = (
                operation_repr[t]
                .unsqueeze(0)
                .expand(
                    len(candidate_indices),
                    -1,
                )
            )

            candidate_repr = operation_repr[
                candidate_t
            ]

            scores = model.score_pairs(
                current_repr,
                candidate_repr,
            ).cpu().numpy()

            # candidate_indices[0] is the reference next operation.
            h1, h3, mrr = expected_tie_metrics(
                scores,
                target_position=0,
            )

            target_signature = (
                test_pack["full_signatures"][t + 1]
            )

            rows.append({
                "position": t,
                "hits1": h1,
                "hits3": h3,
                "mrr": mrr,
                "signature_seen_in_training":
                    target_signature in seen_signatures,
            })

    detail = pd.DataFrame(rows)

    metrics = {
        "hits1": float(detail["hits1"].mean()),
        "hits3": float(detail["hits3"].mean()),
        "mrr": float(detail["mrr"].mean()),
        "seen_signature_rate": float(
            detail["signature_seen_in_training"].mean()
        ),
    }

    seen = detail[
        detail["signature_seen_in_training"]
    ]

    unseen = detail[
        ~detail["signature_seen_in_training"]
    ]

    for label, subset in [
        ("seen", seen),
        ("unseen", unseen),
    ]:
        if len(subset) > 0:
            metrics[f"{label}_hits1"] = float(
                subset["hits1"].mean()
            )

            metrics[f"{label}_hits3"] = float(
                subset["hits3"].mean()
            )

            metrics[f"{label}_mrr"] = float(
                subset["mrr"].mean()
            )
        else:
            metrics[f"{label}_hits1"] = np.nan
            metrics[f"{label}_hits3"] = np.nan
            metrics[f"{label}_mrr"] = np.nan

    return metrics, detail

In [ ]:
# 12. Trainable-capacity helpers

def count_trainable_parameters(model):
    return sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )


def select_capacity_matched_width(vocab):
    gnn_params = count_trainable_parameters(
        build_gnn(vocab)
    )

    candidates = []

    for width in MATCHED_WIDTH_CANDIDATES:
        model = build_mlp(
            vocab,
            hidden_dim=width,
        )
        n_params = count_trainable_parameters(model)
        candidates.append(
            (abs(n_params - gnn_params), width, n_params)
        )
        del model

    _, best_width, best_params = min(candidates)

    relative_gap = (
        abs(best_params - gnn_params) / gnn_params
    )

    return {
        "width": int(best_width),
        "mlp_params": int(best_params),
        "gnn_params": int(gnn_params),
        "relative_gap": float(relative_gap),
    }


In [ ]:
# 13. Smoke test the no-progress sensitivity conditions on one fold.

TEST_BATTERY = 1

train_batteries = [
    b
    for b in sorted(chains)
    if b != TEST_BATTERY
]

vocab = build_fold_vocab(train_batteries)

probe_real = build_gnn(vocab)
probe_noedge = build_gnn(vocab)
probe_shuffle = build_gnn(vocab)

gnn_param_counts = {
    "RealGNN": count_trainable_parameters(probe_real),
    "NoEdgesGNN": count_trainable_parameters(probe_noedge),
    "ShuffledGNN": count_trainable_parameters(probe_shuffle),
}

print("GNN parameter counts:", gnn_param_counts)

assert len(set(gnn_param_counts.values())) == 1, (
    "All graph conditions must have identical parameter counts."
)

matched_info = select_capacity_matched_width(vocab)

print(
    "Matched MLP:",
    f"width={matched_info['width']},",
    f"MLP params={matched_info['mlp_params']:,},",
    f"GNN params={matched_info['gnn_params']:,},",
    f"relative gap={100*matched_info['relative_gap']:.3f}%"
)

# Verify five topology permutations differ for the held-out battery.
edge_fingerprints = []

for sid in SHUFFLE_IDS:
    pack = build_battery_pack(
        TEST_BATTERY,
        vocab,
        graph_mode="shuffle",
        shuffle_id=sid,
    )

    edge_fingerprints.append(
        hash(pack["edge_index"].tobytes())
    )

assert len(set(edge_fingerprints)) == len(SHUFFLE_IDS), (
    "Permutation controls are not all distinct."
)

print("Five distinct topology permutations verified.")

# Short runtime/code smoke test.
old_epochs = EPOCHS
EPOCHS = 2

for label, mode, sid in [
    ("RealGNN", "real", 0),
    ("NoEdgesGNN", "noedge", 0),
    ("ShuffledGNN_1", "shuffle", 1),
]:
    model, pack, _ = train_one_gnn_condition(
        train_batteries,
        TEST_BATTERY,
        seed=SEEDS[0],
        graph_mode=mode,
        shuffle_id=sid,
    )

    metrics, _ = evaluate_neural_model(
        model,
        pack,
        train_batteries,
    )

    print(label, metrics)

model, pack, _ = train_one_mlp_condition(
    train_batteries,
    TEST_BATTERY,
    seed=SEEDS[0],
    hidden_dim=matched_info["width"],
)

metrics, _ = evaluate_neural_model(
    model,
    pack,
    train_batteries,
)

print("MatchedMLP", metrics)

EPOCHS = old_epochs

print("Smoke test passed.")


In [ ]:
# 14. Full no-progress sensitivity experiment (resumable).
#
# Conditions:
#   RealGNN
#   NoEdgesGNN
#   ShuffledGNN with five semantic--topology permutations
#   Capacity-matched MLP
#
# Total: 10 folds x 3 seeds x 8 conditions = 240 fits.

OUT = Path(
    "/content/battery_no_progress_sensitivity_lopo"
)

OUT.mkdir(
    parents=True,
    exist_ok=True,
)

PARTIAL = (
    OUT / "no_progress_sensitivity_partial.csv"
)

if PARTIAL.exists():
    results = pd.read_csv(PARTIAL)
    print(
        "Resuming from",
        len(results),
        "completed fits.",
    )
else:
    results = pd.DataFrame()

battery_ids = sorted(chains)
start_time = time.time()

gnn_conditions = [
    ("RealGNN", "real", 0),
    ("NoEdgesGNN", "noedge", 0),
] + [
    ("ShuffledGNN", "shuffle", sid)
    for sid in SHUFFLE_IDS
]

for fold_idx, test_battery in enumerate(
    battery_ids,
    start=1,
):
    train_batteries = [
        b
        for b in battery_ids
        if b != test_battery
    ]

    vocab = build_fold_vocab(train_batteries)

    matched_info = select_capacity_matched_width(vocab)
    expected_gnn_params = matched_info["gnn_params"]
    matched_width = matched_info["width"]
    matched_params = matched_info["mlp_params"]

    print(
        f"\nFold {fold_idx}/10 — held out: "
        f"{battery_names[test_battery]}"
    )

    print(
        f"  GNN params={expected_gnn_params:,}; "
        f"matched MLP width={matched_width}; "
        f"MLP params={matched_params:,}; "
        f"gap={100*matched_info['relative_gap']:.3f}%"
    )

    # Graph conditions.
    for model_label, graph_mode, shuffle_id in gnn_conditions:
        for seed in SEEDS:
            done = False

            if len(results) > 0:
                mask = (
                    (results["battery_id"] == test_battery)
                    & (results["model"] == model_label)
                    & (results["seed"] == seed)
                    & (results["shuffle_id"] == shuffle_id)
                )
                done = bool(mask.any())

            if done:
                print(
                    f"  skip {model_label} "
                    f"shuffle={shuffle_id} seed={seed}"
                )
                continue

            model, test_pack, _ = train_one_gnn_condition(
                train_batteries,
                test_battery,
                seed=seed,
                graph_mode=graph_mode,
                shuffle_id=shuffle_id,
            )

            metrics, _ = evaluate_neural_model(
                model,
                test_pack,
                train_batteries,
            )

            param_count = count_trainable_parameters(model)
            assert param_count == expected_gnn_params

            row = {
                "battery_id": test_battery,
                "battery": battery_names[test_battery],
                "model": model_label,
                "graph_mode": graph_mode,
                "shuffle_id": int(shuffle_id),
                "seed": int(seed),
                "hidden_width": HIDDEN_DIM,
                "hits1": metrics["hits1"],
                "hits3": metrics["hits3"],
                "mrr": metrics["mrr"],
                "parameter_count": param_count,
                "gnn_parameter_count": expected_gnn_params,
                "matched_parameter_gap": matched_info["relative_gap"],
            }

            results = pd.concat(
                [results, pd.DataFrame([row])],
                ignore_index=True,
            )

            results.to_csv(
                PARTIAL,
                index=False,
            )

            print(
                f"  {model_label} "
                f"shuffle={shuffle_id} seed={seed}: "
                f"H@1={metrics['hits1']:.3f}, "
                f"H@3={metrics['hits3']:.3f}, "
                f"MRR={metrics['mrr']:.3f}"
            )

    # Capacity-matched MLP.
    for seed in SEEDS:
        model_label = "MatchedMLP"
        shuffle_id = 0

        done = False

        if len(results) > 0:
            mask = (
                (results["battery_id"] == test_battery)
                & (results["model"] == model_label)
                & (results["seed"] == seed)
                & (results["shuffle_id"] == shuffle_id)
            )
            done = bool(mask.any())

        if done:
            print(
                f"  skip {model_label} seed={seed}"
            )
            continue

        model, test_pack, _ = train_one_mlp_condition(
            train_batteries,
            test_battery,
            seed=seed,
            hidden_dim=matched_width,
        )

        metrics, _ = evaluate_neural_model(
            model,
            test_pack,
            train_batteries,
        )

        param_count = count_trainable_parameters(model)
        assert param_count == matched_params

        row = {
            "battery_id": test_battery,
            "battery": battery_names[test_battery],
            "model": model_label,
            "graph_mode": "none",
            "shuffle_id": 0,
            "seed": int(seed),
            "hidden_width": matched_width,
            "hits1": metrics["hits1"],
            "hits3": metrics["hits3"],
            "mrr": metrics["mrr"],
            "parameter_count": param_count,
            "gnn_parameter_count": expected_gnn_params,
            "matched_parameter_gap": matched_info["relative_gap"],
        }

        results = pd.concat(
            [results, pd.DataFrame([row])],
            ignore_index=True,
        )

        results.to_csv(
            PARTIAL,
            index=False,
        )

        print(
            f"  {model_label} width={matched_width} seed={seed}: "
            f"H@1={metrics['hits1']:.3f}, "
            f"H@3={metrics['hits3']:.3f}, "
            f"MRR={metrics['mrr']:.3f}"
        )

elapsed = time.time() - start_time

print(
    f"\nCompleted/resumed to {len(results)} fits "
    f"in {elapsed/60:.1f} minutes."
)


In [ ]:
# 15. Build paired seed-level and held-out-product summaries.

real_seed = (
    results[
        results["model"] == "RealGNN"
    ][
        ["battery_id", "battery", "seed", "hits1", "hits3", "mrr"]
    ]
    .rename(
        columns={
            "hits1": "real_hits1",
            "hits3": "real_hits3",
            "mrr": "real_mrr",
        }
    )
)

noedge_seed = (
    results[
        results["model"] == "NoEdgesGNN"
    ][
        ["battery_id", "battery", "seed", "hits1", "hits3", "mrr"]
    ]
    .rename(
        columns={
            "hits1": "noedge_hits1",
            "hits3": "noedge_hits3",
            "mrr": "noedge_mrr",
        }
    )
)

shuffle_seed = (
    results[
        results["model"] == "ShuffledGNN"
    ]
    .groupby(
        ["battery_id", "battery", "seed"],
        as_index=False,
    )
    .agg(
        shuffle_hits1=("hits1", "mean"),
        shuffle_hits3=("hits3", "mean"),
        shuffle_mrr=("mrr", "mean"),
        shuffle_hits1_sd=("hits1", "std"),
    )
)

matched_seed = (
    results[
        results["model"] == "MatchedMLP"
    ][
        ["battery_id", "battery", "seed", "hits1", "hits3", "mrr"]
    ]
    .rename(
        columns={
            "hits1": "matched_hits1",
            "hits3": "matched_hits3",
            "mrr": "matched_mrr",
        }
    )
)

paired_seed = (
    real_seed
    .merge(
        noedge_seed,
        on=["battery_id", "battery", "seed"],
        how="inner",
    )
    .merge(
        shuffle_seed,
        on=["battery_id", "battery", "seed"],
        how="inner",
    )
    .merge(
        matched_seed,
        on=["battery_id", "battery", "seed"],
        how="inner",
    )
)

for metric in ["hits1", "hits3", "mrr"]:
    paired_seed[f"real_minus_shuffle_{metric}"] = (
        paired_seed[f"real_{metric}"]
        - paired_seed[f"shuffle_{metric}"]
    )
    paired_seed[f"real_minus_noedge_{metric}"] = (
        paired_seed[f"real_{metric}"]
        - paired_seed[f"noedge_{metric}"]
    )
    paired_seed[f"shuffle_minus_noedge_{metric}"] = (
        paired_seed[f"shuffle_{metric}"]
        - paired_seed[f"noedge_{metric}"]
    )
    paired_seed[f"real_minus_matched_{metric}"] = (
        paired_seed[f"real_{metric}"]
        - paired_seed[f"matched_{metric}"]
    )

# Average repeated neural seeds within each held-out battery.
fold_mean = (
    paired_seed
    .groupby(
        ["battery_id", "battery"],
        as_index=False,
    )
    .agg(
        real_hits1=("real_hits1", "mean"),
        shuffle_hits1=("shuffle_hits1", "mean"),
        noedge_hits1=("noedge_hits1", "mean"),
        matched_hits1=("matched_hits1", "mean"),
        real_hits3=("real_hits3", "mean"),
        shuffle_hits3=("shuffle_hits3", "mean"),
        noedge_hits3=("noedge_hits3", "mean"),
        matched_hits3=("matched_hits3", "mean"),
        real_mrr=("real_mrr", "mean"),
        shuffle_mrr=("shuffle_mrr", "mean"),
        noedge_mrr=("noedge_mrr", "mean"),
        matched_mrr=("matched_mrr", "mean"),
        shuffle_hits1_sd=("shuffle_hits1_sd", "mean"),
    )
)

for metric in ["hits1", "hits3", "mrr"]:
    fold_mean[f"real_minus_shuffle_{metric}"] = (
        fold_mean[f"real_{metric}"]
        - fold_mean[f"shuffle_{metric}"]
    )
    fold_mean[f"real_minus_noedge_{metric}"] = (
        fold_mean[f"real_{metric}"]
        - fold_mean[f"noedge_{metric}"]
    )
    fold_mean[f"shuffle_minus_noedge_{metric}"] = (
        fold_mean[f"shuffle_{metric}"]
        - fold_mean[f"noedge_{metric}"]
    )
    fold_mean[f"real_minus_matched_{metric}"] = (
        fold_mean[f"real_{metric}"]
        - fold_mean[f"matched_{metric}"]
    )

display(
    fold_mean.sort_values(
        "real_minus_shuffle_hits1",
        ascending=False,
    )
)


In [ ]:
# 16. Aggregate no-progress results and exact paired sign-flip tests.

def exact_signflip_p(values):
    values = np.asarray(values, dtype=float)
    observed = abs(values.mean())

    signs = np.asarray(
        list(product([-1.0, 1.0], repeat=len(values)))
    )

    null_means = np.abs(
        (signs * values).mean(axis=1)
    )

    return float(
        np.mean(null_means >= observed - 1e-15)
    )


def holm_adjust(p_values):
    p_values = np.asarray(p_values, dtype=float)
    order = np.argsort(p_values)
    adjusted = np.empty_like(p_values)
    running = 0.0
    m = len(p_values)

    for rank, idx in enumerate(order):
        candidate = (m - rank) * p_values[idx]
        running = max(running, candidate)
        adjusted[idx] = min(1.0, running)

    return adjusted


summary = pd.DataFrame([
    {
        "condition": "RealGNN",
        "mean_hits1": fold_mean["real_hits1"].mean(),
        "mean_hits3": fold_mean["real_hits3"].mean(),
        "mean_mrr": fold_mean["real_mrr"].mean(),
    },
    {
        "condition": "FivePermutationMean",
        "mean_hits1": fold_mean["shuffle_hits1"].mean(),
        "mean_hits3": fold_mean["shuffle_hits3"].mean(),
        "mean_mrr": fold_mean["shuffle_mrr"].mean(),
    },
    {
        "condition": "NoEdgesGNN",
        "mean_hits1": fold_mean["noedge_hits1"].mean(),
        "mean_hits3": fold_mean["noedge_hits3"].mean(),
        "mean_mrr": fold_mean["noedge_mrr"].mean(),
    },
    {
        "condition": "MatchedMLP",
        "mean_hits1": fold_mean["matched_hits1"].mean(),
        "mean_hits3": fold_mean["matched_hits3"].mean(),
        "mean_mrr": fold_mean["matched_mrr"].mean(),
    },
])

display(summary)

comparison_specs = [
    ("Real - Permuted", "real", "shuffle"),
    ("Real - No-edge", "real", "noedge"),
    ("Permuted - No-edge", "shuffle", "noedge"),
    ("Real - MatchedMLP", "real", "matched"),
]

rows = []

for metric in ["hits1", "hits3", "mrr"]:
    for label, a, b in comparison_specs:
        d = (
            fold_mean[f"{a}_{metric}"]
            - fold_mean[f"{b}_{metric}"]
        )

        rows.append({
            "metric": metric,
            "comparison": label,
            "mean_difference": float(d.mean()),
            "wins": int((d > 0).sum()),
            "n_products": len(d),
            "p_exact": exact_signflip_p(d),
        })

exact_tests = pd.DataFrame(rows)

# Holm adjustment for the three topology contrasts on primary Hits@1.
primary_mask = (
    (exact_tests["metric"] == "hits1")
    & exact_tests["comparison"].isin([
        "Real - Permuted",
        "Real - No-edge",
        "Permuted - No-edge",
    ])
)

primary_p = exact_tests.loc[
    primary_mask,
    "p_exact",
].to_numpy()

exact_tests.loc[
    primary_mask,
    "p_holm_hits1_topology",
] = holm_adjust(primary_p)

display(exact_tests)


In [ ]:
# 17. Product-level bootstrap intervals and final paired statistics.

def bootstrap_mean_ci(
    values,
    n_boot=20000,
    seed=20261201,
):
    values = np.asarray(values, dtype=float)
    rng = np.random.default_rng(seed)

    draws = rng.choice(
        values,
        size=(n_boot, len(values)),
        replace=True,
    ).mean(axis=1)

    return (
        float(np.percentile(draws, 2.5)),
        float(np.percentile(draws, 97.5)),
    )


bootstrap_seeds = {
    ("hits1", "Real - Permuted"): 20261201,
    ("hits1", "Real - No-edge"): 20261202,
    ("hits1", "Permuted - No-edge"): 20261203,
    ("hits1", "Real - MatchedMLP"): 20261204,
    ("hits3", "Real - Permuted"): 20261211,
    ("hits3", "Real - No-edge"): 20261212,
    ("hits3", "Permuted - No-edge"): 20261213,
    ("hits3", "Real - MatchedMLP"): 20261214,
    ("mrr", "Real - Permuted"): 20261221,
    ("mrr", "Real - No-edge"): 20261222,
    ("mrr", "Permuted - No-edge"): 20261223,
    ("mrr", "Real - MatchedMLP"): 20261224,
}

contrast_columns = {
    "Real - Permuted": ("real", "shuffle"),
    "Real - No-edge": ("real", "noedge"),
    "Permuted - No-edge": ("shuffle", "noedge"),
    "Real - MatchedMLP": ("real", "matched"),
}

paired_rows = []

for metric in ["hits1", "hits3", "mrr"]:
    for label, (a, b) in contrast_columns.items():
        d = (
            fold_mean[f"{a}_{metric}"]
            - fold_mean[f"{b}_{metric}"]
        )

        lo, hi = bootstrap_mean_ci(
            d,
            seed=bootstrap_seeds[(metric, label)],
        )

        p = exact_signflip_p(d)

        paired_rows.append({
            "metric": metric,
            "comparison": label,
            "mean_difference": float(d.mean()),
            "ci_low": lo,
            "ci_high": hi,
            "wins": int((d > 0).sum()),
            "n_products": len(d),
            "p_exact": p,
        })

paired_stats = pd.DataFrame(paired_rows)

primary_mask = (
    (paired_stats["metric"] == "hits1")
    & paired_stats["comparison"].isin([
        "Real - Permuted",
        "Real - No-edge",
        "Permuted - No-edge",
    ])
)

paired_stats.loc[
    primary_mask,
    "p_holm_hits1_topology",
] = holm_adjust(
    paired_stats.loc[
        primary_mask,
        "p_exact",
    ].to_numpy()
)

display(paired_stats)

print("\nPrimary qualitative mechanism check:")

for label in [
    "Real - No-edge",
    "Permuted - No-edge",
    "Real - Permuted",
    "Real - MatchedMLP",
]:
    row = paired_stats[
        (paired_stats["metric"] == "hits1")
        & (paired_stats["comparison"] == label)
    ].iloc[0]

    print(
        f"{label}: "
        f"mean diff={row['mean_difference']:.4f}, "
        f"wins={int(row['wins'])}/10, "
        f"95% CI=({row['ci_low']:.4f}, {row['ci_high']:.4f}), "
        f"exact p={row['p_exact']:.6f}"
    )


In [ ]:
# 18. Save all no-progress sensitivity outputs.

FINAL_OUT = Path(
    "/content/battery_no_progress_sensitivity_lopo"
)

FINAL_OUT.mkdir(
    parents=True,
    exist_ok=True,
)

results.to_csv(
    FINAL_OUT
    / "no_progress_sensitivity_seed_results.csv",
    index=False,
)

paired_seed.to_csv(
    FINAL_OUT
    / "no_progress_sensitivity_paired_seed.csv",
    index=False,
)

fold_mean.to_csv(
    FINAL_OUT
    / "no_progress_sensitivity_fold_mean.csv",
    index=False,
)

summary.to_csv(
    FINAL_OUT
    / "no_progress_sensitivity_summary.csv",
    index=False,
)

paired_stats.to_csv(
    FINAL_OUT
    / "no_progress_sensitivity_paired.csv",
    index=False,
)

print("Saved to:", FINAL_OUT)

for p in sorted(FINAL_OUT.iterdir()):
    print(" -", p.name)


## What to send back

After the full run finishes, please upload these three files:

- `no_progress_sensitivity_seed_results.csv`
- `no_progress_sensitivity_fold_mean.csv`
- `no_progress_sensitivity_paired.csv`

The optional `no_progress_sensitivity_summary.csv` and `no_progress_sensitivity_paired_seed.csv` are also useful.

Do not replace the canonical `final_topology_mechanism_*` files with these outputs. This notebook is a sensitivity analysis whose purpose is to test whether the manuscript's qualitative mechanism conclusion survives removal of sequence progress.
